# Separate attention pattern, values, and output

Measure attention mass on newline keys and inspect the head-1 pattern.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/06_layer1_mechanisms"))
from common import *
EXAMPLE_SEED=6001
ctx=load_context(EXAMPLE_SEED)
model,vocab,batch,cfg,device=ctx.model,ctx.vocab,ctx.batch,ctx.cfg,ctx.device
tokens=batch.tokens[0].detach().cpu().numpy()
print("checkpoint",ctx.checkpoint["step"],"seed",EXAMPLE_SEED,"line width",int(batch.line_lengths[0]))
print("newline positions",newline_positions(tokens,vocab))

/Users/rohanbuluswar/Desktop/Line Break Project/.venv_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vocab...
('BOS', '_NEWLINE_')
checkpoint 10000 seed 6001 line width 89
newline positions [22, 44, 66, 87, 106, 129, 149]


In [2]:
names=["blocks.1.attn.hook_pattern","blocks.1.attn.hook_v","blocks.1.attn.hook_z"]
_,c=model.run_with_cache(batch.tokens,prepend_bos=False,names_filter=names)
pattern=c[names[0]][0,1].detach().cpu().numpy(); v=c[names[1]][0,:,1].detach().cpu().numpy(); z=c[names[2]][0,:,1].detach().cpu().numpy()
nls=newline_positions(tokens,vocab); mass=pattern[:,nls].sum(axis=1)
print("mean newline-key attention mass",float(mass.mean()))
print("mean value norm",float(np.linalg.norm(v,axis=1).mean()),"mean output norm",float(np.linalg.norm(z,axis=1).mean()))
plot_heatmap(pattern.T,"Layer 1 head 1 attention pattern","figures/layer1_head1_pattern.png")

mean newline-key attention mass 0.5152929425239563
mean value norm 7.862441539764404 mean output norm 4.56955099105835
